# 14. Comparación de Modelos

## 14.1. Criterios de información (modelos paramétricos)

Los criterios AIC, AICc y BIC penalizan la verosimilitud por la complejidad del modelo. El **AICc** es la métrica preferida cuando n < 40.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
wv_acc = (estados[(estados['state']=='West Virginia')&
                    (estados['cause_name']=='Unintentional injuries')]
           .sort_values('year').dropna(subset=['age_adjusted_death_rate']))
serie = wv_acc.set_index('year')['age_adjusted_death_rate']

# Ajustar modelos
mejor_aicc, mejor_orden, mejor_modelo = np.inf, None, None
for p in range(0,3):
    for d in range(0,2):
        for q in range(0,3):
            try:
                m = ARIMA(serie, order=(p,d,q)).fit()
                if m.aicc < mejor_aicc:
                    mejor_aicc, mejor_orden, mejor_modelo = m.aicc, (p,d,q), m
            except: pass

ets = ExponentialSmoothing(serie, trend='add').fit(optimized=True)
ets_holt = ExponentialSmoothing(serie, trend='add').fit()
ets_damp = ExponentialSmoothing(serie, trend='add', damped_trend=True).fit()

tabla_aic = pd.DataFrame({
    'Modelo': [f'ARIMA{mejor_orden} with drift',
               f'ETS – {ets.model.__class__.__name__}',
               'Holt lineal – ETS(A,A,N)',
               'Holt amortiguado – ETS(A,Ad,N)'],
    'AIC':  [round(mejor_modelo.aic,2), round(ets.aic,2), round(ets_holt.aic,2), round(ets_damp.aic,2)],
    'AICc': [round(mejor_modelo.aicc,2), round(ets.aicc,2), round(ets_holt.aicc,2), round(ets_damp.aicc,2)],
    'BIC':  [round(mejor_modelo.bic,2), round(ets.bic,2), round(ets_holt.bic,2), round(ets_damp.bic,2)]
})
print("Criterios de información – modelos paramétricos (n=19):")
print(tabla_aic.to_string(index=False))
print(f"\nMejor modelo por AICc: {tabla_aic.loc[tabla_aic['AICc'].idxmin(),'Modelo']}")

Criterios de información – modelos paramétricos (n=19):
                        Modelo    AIC   AICc    BIC
     ARIMA(0, 1, 0) with drift 130.41 130.66 131.30
    ETS – ExponentialSmoothing  79.47  86.47  83.24
      Holt lineal – ETS(A,A,N)  79.47  86.47  83.24
Holt amortiguado – ETS(A,Ad,N)  81.84  92.02  86.56

Mejor modelo por AICc: ETS – ExponentialSmoothing


C:\Users\luisc\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\luisc\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 14.2. Diagnóstico comparativo de residuos

In [3]:
resid_arima = mejor_modelo.resid
resid_ets = ets.resid
serie_arr = serie.values
X = np.arange(1, len(serie_arr)+1).reshape(-1,1)
lm = LinearRegression().fit(X, serie_arr)
resid_lm = serie_arr - lm.predict(X)

for name, resid in [('ARIMA', resid_arima),('ETS', resid_ets),('Regresión Lineal', resid_lm)]:
    lb = acorr_ljungbox(resid, lags=[4], return_df=True)
    print(f"{name}: Q*={lb['lb_stat'].values[0]:.3f}, p={lb['lb_pvalue'].values[0]:.4f} "
          f"→ {'✔ Independiente' if lb['lb_pvalue'].values[0]>0.05 else '⚠ Autocorrelación'}")

ARIMA: Q*=1.313, p=0.8591 → ✔ Independiente
ETS: Q*=4.770, p=0.3117 → ✔ Independiente
Regresión Lineal: Q*=4.770, p=0.3117 → ✔ Independiente


## 14.3. Validación cruzada de series de tiempo (tsCV)

In [4]:
def tscv(serie, modelo_fn, h=1, min_obs=10):
    errores = []
    for i in range(min_obs, len(serie)-h+1):
        train = serie[:i]
        actual = serie[i:i+h]
        try:
            pred = modelo_fn(train, h)
            errores.append(np.mean((actual.values - pred)**2))
        except: pass
    return np.sqrt(np.mean(errores)) if errores else np.nan

def arima_fn(s, h): return ARIMA(s, order=mejor_orden).fit().forecast(h)
def lm_fn(s, h):
    X = np.arange(1,len(s)+1).reshape(-1,1)
    m = LinearRegression().fit(X, s.values)
    Xf = np.arange(len(s)+1, len(s)+h+1).reshape(-1,1)
    return m.predict(Xf)

rmse_arima = tscv(serie, arima_fn, h=1)
rmse_lm = tscv(serie, lm_fn, h=1)

print("Validación cruzada (tsCV) – RMSE h=1:")
print(f"  ARIMA:            {rmse_arima:.3f}")
print(f"  Regresión Lineal: {rmse_lm:.3f}")

Validación cruzada (tsCV) – RMSE h=1:
  ARIMA:            10.431
  Regresión Lineal: 10.352


## 14.4. Gráfico comparativo de proyecciones

In [5]:
anios_fut = list(range(2018,2023))
fc_arima = mejor_modelo.get_forecast(5).predicted_mean
fc_ets_vals = ets.forecast(5)
X_fut = np.arange(len(serie)+1, len(serie)+6).reshape(-1,1)
fc_lm = lm.predict(X_fut)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(serie.index), y=serie.values, mode='lines+markers',
                          name='Observado', line=dict(color='#1D3557', width=2.5)))
fig.add_trace(go.Scatter(x=anios_fut, y=fc_arima.values, mode='lines+markers',
                          name=f'ARIMA {mejor_orden}', line=dict(color='#E63946', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=anios_fut, y=fc_ets_vals.values, mode='lines+markers',
                          name='ETS', line=dict(color='#27ae60', width=2, dash='dot')))
fig.add_trace(go.Scatter(x=anios_fut, y=fc_lm, mode='lines+markers',
                          name='Regresión Lineal', line=dict(color='#6A4C93', width=2, dash='longdash')))
fig.add_vline(x=2017.5, line_dash='dot', line_color='gray',
              annotation_text='→ Proyección', annotation_position='top right')
fig.update_layout(title='Comparación de proyecciones – Unintentional Injuries · West Virginia (2018–2022)',
                   xaxis_title='Año', yaxis_title='Tasa por 100,000 hab.',
                   height=480, template='plotly_white',
                   xaxis=dict(tickmode='linear', tick0=1999, dtick=2))
fig.show()